# Modular Expansion with Unit Commitment (MILP)

Self-contained PyPSA + HiGHS analysis of extendable, committable assets that
can only be built in fixed-size modules. Adapted from the PyPSA example
notebook *Modular Expansion with Unit Commitment* (PyPSA contributors,
[CC-BY-4.0](https://creativecommons.org/licenses/by/4.0/)).

The reusable analysis lives in the local module `milp_analysis.py` in
PROJECT_ROOT. This notebook runs from any directory: `PROJECT_ROOT` is
supplied via the environment (the generated project is on `sys.path`).

In [ ]:
import json
import os
import sys

PROJECT_ROOT = os.environ.get("PROJECT_ROOT")
if PROJECT_ROOT:
    sys.path.insert(0, PROJECT_ROOT)

import energy_core as core

print("Scenario keys:", list(core.default_settings().keys()))


In [ ]:
# Reference case: the hand-checkable 21879 optimum on the base four-hour profile.
result = core.solve_scenario(core.default_settings())
print("Status:", result["status"])
print("Optimal capacity [MW]:", result["capacity_mw"])
print("Modules built:", result["modules"])
print("Objective:", result["objective"])
print("Active modules per hour:", result["active_modules"])


In [ ]:
# Solve the stable scientific scenarios and write results.json in the current directory.
base = core.default_settings()
summary = {
    "reference": core.solve_scenario(base),
    "solar": core.solve_scenario(base | {"solar_capacity": 1500.0, "startup_cost": 500.0}),
    "startup": core.solve_scenario(base | {"hours": 12, "startup_cost": 500.0}),
    "shortage": core.solve_scenario(base | {"max_modules": 20, "allow_shedding": True}),
}
with open(os.path.join(os.getcwd(), "results.json"), "w") as fh:
    json.dump({"results": summary}, fh, indent=2, allow_nan=False)
print("Wrote results.json with", len(summary), "cases")
